# 04 — Single-line-to-ground fault

## Objectives and engineering concept

Apply a declared single-line-to-ground fault and distinguish a solver current from a protection-duty decision.

## System, assumptions, and units

The IEEE13 feeder is faulted at phase 1 of bus 675 with `rf=0.001 ohm`. Current is in A and voltage is in pu.

## Part A — Pure OpenDSS

The fault element is created, the network is solved, and its current is read back.

## Part B — Same study with CEPT

CEPT expresses the same fault in typed study options and retains the solver-returned result.

## Part C — Compare and verify

The fault-element current magnitudes are compared with a declared teaching tolerance. The manual setup is the repetition CEPT automates; it does not certify protection settings or field short-circuit duty.

We apply the same bolted-network fault resistance (`rf=0.001 ohm`) at phase 1 of bus 675 in direct OpenDSS and through CEPT. The quantity compared is the current magnitude returned by the solver's fault element.

## Interpret, exercise, and reproduce

Interpret the current with the stated source and feeder assumptions. As an exercise, compare another fault type only when its phase semantics are explicit; restart and run all cells to reproduce the displayed solver-backed result.

In [ ]:
from importlib.resources import files
from pathlib import Path
import opendssdirect as dss
from cept.public import demo_case, run_study

master = Path(str(files('cept').joinpath('testsystems', 'ieee13', 'IEEE13Nodeckt.dss')))
dss.Basic.ClearAll()
dss.Basic.DataPath(str(master.parent))
dss.Text.Command(f'Redirect "{master}"')
dss.Text.Command('New Fault.lesson_fault Bus1=675.1 phases=1 r=0.001')
dss.Text.Command('Solve')
assert dss.Solution.Converged()
dss.Circuit.SetActiveElement('Fault.lesson_fault')
currents = dss.CktElement.Currents()
direct_current_a = abs(complex(currents[0], currents[1]))

run = run_study(demo_case('fault'))
fault = run.result.fault
assert run.verification['passed'] is True
cept_current_a = float(fault.total_fault_current_a)
print({'direct_current_a': direct_current_a, 'cept_current_a': cept_current_a})
assert abs(direct_current_a - cept_current_a) < 1.0


Fault current depends on the declared feeder and source model. The lesson verifies the public translation against the same solver and inputs; it does not certify protection settings or a field short-circuit duty.